# Encoder: XLM-R full, second epoch
Continues `xlmr_full` (output of `hw3-encoder-full`) for one more epoch over the whole train fold with a lower LR (1e-5) and a new shuffle seed. Everything else is identical; val fold is untouched.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

SEED = 42
MAX_LEN = 512
HEAD = 128
LABELS = ['human', 'ai', 'fusion']
L2I = {l: i for i, l in enumerate(LABELS)}

set_seed(SEED)
INPUT = Path('/kaggle/input')
MODEL = next(INPUT.rglob('xlmr_full/config.json')).parent
train = pd.read_parquet(next(INPUT.rglob('train_clean.parquet')), columns=['id', 'text', 'lang', 'domain', 'label', 'fold'])
test = pd.read_parquet(next(INPUT.rglob('test_clean.parquet')))
sample_sub = pd.read_csv(next(INPUT.rglob('sample_submission.csv')))

tr = train[train.fold == 'train']
va = train[train.fold == 'val']
del train
len(tr), len(va), len(test), torch.cuda.device_count()

## Head+tail tokenization
Texts longer than 510 tokens keep the first 128 and the last 382: fusion rewrites are mostly after the beginning.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
tok.model_max_length = 10**6

def encode(texts, chunk=10_000):
    body = MAX_LEN - 2
    out = []
    for i in range(0, len(texts), chunk):
        for x in tok(list(texts[i:i + chunk]), add_special_tokens=False, return_attention_mask=False)['input_ids']:
            if len(x) > body:
                x = x[:HEAD] + x[len(x) - (body - HEAD):]
            out.append(np.array([tok.cls_token_id] + x + [tok.sep_token_id], np.int32))
    return out

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.ids = encode(df.text)
        self.labels = df.label.map(L2I).tolist() if 'label' in df else None

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        item = {'input_ids': self.ids[i].tolist()}
        if self.labels is not None:
            item['labels'] = self.labels[i]
        return item

ds_tr, ds_va, ds_te = TextDataset(tr), TextDataset(va), TextDataset(test)
tr, va, test = (d.drop(columns='text') for d in (tr, va, test))
ds_mon = torch.utils.data.Subset(ds_va, np.random.RandomState(SEED).choice(len(ds_va), min(5000, len(ds_va)), replace=False))
pd.Series([len(x) for x in ds_tr.ids]).describe().round(0)

## Training

In [ ]:
def macro_f1(p):
    return {'macro_f1': f1_score(p.label_ids, p.predictions.argmax(-1), average='macro')}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I, dtype=torch.float32)

BATCH, ACCUM, EPOCHS = 8, 2, 1
steps = int(np.ceil(len(ds_tr) / (BATCH * ACCUM * max(torch.cuda.device_count(), 1)))) * EPOCHS

args = TrainingArguments(
    output_dir='out',
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM,
    learning_rate=1e-5,
    num_train_epochs=EPOCHS,
    warmup_steps=int(.06 * steps),
    weight_decay=.01,
    lr_scheduler_type='linear',
    fp16=torch.cuda.is_available(),
    eval_strategy='steps',
    eval_steps=1500,
    logging_steps=100,
    save_strategy='no',
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED + 1,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_mon,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=macro_f1)
trainer.train()
trainer.save_model('xlmr_full_ep2')
tok.save_pretrained('xlmr_full_ep2')

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
log.dropna(subset=['loss']).plot(x='step', y='loss', ax=ax[0], legend=False, title='train loss', color='#2a78d6')
log.dropna(subset=['eval_macro_f1']).plot(x='step', y='eval_macro_f1', ax=ax[1], legend=False, title='monitor macro-F1 (5k val)', marker='o', color='#2a78d6')
for a in ax:
    a.spines[['top', 'right']].set_visible(False)
    a.grid(alpha=.3)
plt.tight_layout()
plt.show()

## Validation (full val fold)

In [ ]:
va_logits = trainer.predict(ds_va).predictions
va_pred = np.array(LABELS)[va_logits.argmax(-1)]
print(classification_report(va.label, va_pred, labels=LABELS, digits=4))
ConfusionMatrixDisplay.from_predictions(va.label, va_pred, labels=LABELS, normalize='true', cmap='Blues', values_format='.2f')
plt.title('XLM-R full ep2, val')
plt.show()

## Comparison with baseline and epoch 1
Same val ids. Uses `val_scores_baseline.csv` and `val_logits_xlmr_full.csv` if attached.

In [ ]:
preds = {'xlmr_full_ep2': pd.Series(va_pred, index=va.id)}
for f in list(INPUT.rglob('val_scores_baseline.csv')) + list(INPUT.rglob('val_logits_xlmr_full.csv')):
    preds[f.stem[4:]] = pd.read_csv(f, index_col='id').loc[va.id][LABELS].idxmax(axis=1)

y = va.set_index('id').label
def scores(p):
    f = f1_score(y, p.loc[y.index], labels=LABELS, average=None)
    return {**dict(zip(LABELS, f)), 'macro': f.mean()}

pd.DataFrame({k: scores(p) for k, p in preds.items()}).T.round(4)

In [ ]:
v = va.set_index('id')
pd.DataFrame({
    k: pd.concat({by: v.assign(pred=p.loc[v.index]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
                  for by in ['lang', 'domain']})
    for k, p in preds.items()
}).round(4)

## Save logits and submission

In [ ]:
te_logits = trainer.predict(ds_te).predictions
pd.DataFrame(va_logits, columns=LABELS, index=va.id).to_csv('val_logits_xlmr_full_ep2.csv')
pd.DataFrame(te_logits, columns=LABELS, index=test.id).to_csv('test_logits_xlmr_full_ep2.csv')

pred = pd.Series(np.array(LABELS)[te_logits.argmax(-1)], index=test.id)
sample_sub.assign(label=pred.loc[sample_sub.id].values).to_csv('submission_xlmr_full_ep2.csv', index=False)
pd.read_csv('submission_xlmr_full_ep2.csv').label.value_counts(normalize=True).round(3)